# Apriori-TID · Phần bàn giao của Phát

Demo chạy hai bản Apriori-TID trên dữ liệu nhỏ, trình bày benchmark đã đo và tái tạo kết quả phân tích giỏ hàng từ dữ liệu Groceries gốc. Các bước khai thác tập mục và sinh luật chạy bằng Julia.

In [ ]:
import csv
import os
from pathlib import Path
import shutil
import subprocess
import tempfile
from urllib.request import urlretrieve

from IPython.display import Image, Markdown, display

repo_url = os.environ.get("APRIORI_REPO_URL", "https://github.com/torusama/apriori-tid-mining")
repo_ref = os.environ.get("APRIORI_REPO_REF", "main")
repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "run_market_basket.jl").is_file()
             and (p / "Project.toml").is_file()), None)

if repo is None:
    repo = Path.cwd() / "apriori-tid-mining"
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", repo_ref, repo_url, str(repo)], check=True)

julia = shutil.which("julia")
if julia is None:
    version = "1.10.10"
    julia_dir = Path(tempfile.gettempdir()) / ("julia-" + version)
    julia = str(julia_dir / "bin" / "julia")
    if not Path(julia).is_file():
        with tempfile.TemporaryDirectory() as download_dir:
            archive = Path(download_dir) / "julia.tar.gz"
            urlretrieve(f"https://julialang-s3.julialang.org/bin/linux/x64/1.10/julia-{version}-linux-x86_64.tar.gz", archive)
            subprocess.run(["tar", "-xzf", str(archive), "-C", str(julia_dir.parent)], check=True)

workspace = Path(tempfile.mkdtemp(prefix="apriori-phat-demo-"))
raw = workspace / "data"
project = [julia, "--startup-file=no", f"--project={repo}"]
print(subprocess.check_output([julia, "--version"], text=True).strip())

## Kiểm tra hai phiên bản

Hai lần chạy dùng cùng giao dịch và cùng minsup. Kết quả phải trùng cả tập mục và support count.

In [ ]:
for implementation in ("basic", "optimized"):
    subprocess.run(project + [
        str(repo / "scripts" / "cli.jl"),
        "--input", str(repo / "data" / "toy" / "sample.txt"),
        "--output", str(workspace / f"{implementation}.txt"),
        "--minsup", "0.6", "--implementation", implementation,
    ], cwd=repo, check=True)
basic = (workspace / "basic.txt").read_text()
assert basic == (workspace / "optimized.txt").read_text()
print(basic)

## Benchmark đối chiếu với SPMF

Số liệu bên dưới được tạo bằng script benchmark của repo. Runtime Julia là median các lượt đo có warm-up. Cột SPMF dùng thời gian nội bộ do SPMF báo cáo. Cách đo và điều kiện máy được lưu kèm CSV.

In [ ]:
benchmark = repo / "result" / "benchmark"
with (benchmark / "benchmarks.csv").open(newline="") as handle:
    runs = list(csv.DictReader(handle))
datasets = list(dict.fromkeys(row["dataset"] for row in runs))
assert len(datasets) == 4
for dataset in datasets:
    rows = [row for row in runs if row["dataset"] == dataset and row["implementation"] != "spmf"]
    passed = sum(row["exact_match"] == "true" for row in rows)
    print(f"{dataset}: {passed}/{len(rows)} lượt Julia khớp SPMF")
for name in ("runtime_ms.png", "itemsets.png", "peak_ram.png", "scaling_and_width.png"):
    display(Image(filename=benchmark / name))

## Groceries

Mỗi dòng của ma trận arules là một giỏ. Một sản phẩm được tính một lần khi số lượng dương. Demo dùng minsup 1% và minconf 40%, sau đó sắp luật giảm dần theo lift.

In [ ]:
import sys
subprocess.run([sys.executable, str(repo / "scripts" / "fetch_datasets.py"), "groceries", "--root", str(raw)],
               cwd=repo, check=True)
output = workspace / "groceries"
subprocess.run(project + [
    str(repo / "scripts" / "run_market_basket.jl"),
    str(raw / "groceries" / "Groceries.data"),
    str(raw / "groceries" / "Groceries.labels"),
    str(output), "0.01", "0.4",
], cwd=repo, check=True)

with (output / "top10_rules.csv").open(newline="") as handle:
    rules = list(csv.DictReader(handle))
assert len(rules) == 10
table = ["| Vế trái | Vế phải | Số giỏ | Support | Confidence | Lift |", "|---|---|---:|---:|---:|---:|"]
for row in rules:
    table.append(f'| {row["antecedent"]} | {row["consequent"]} | {row["support_count"]} | '
                 f'{float(row["support"]):.2%} | {float(row["confidence"]):.2%} | {float(row["lift"]):.2f} |')
display(Markdown("\n".join(table)))
display(Image(filename=repo / "result" / "market_basket" / "top10_rules.png"))

## Diễn giải

Luật citrus fruit + root vegetables → other vegetables đứng đầu về lift, với 102 giỏ thỏa cả hai vế. Luật root vegetables + whole milk → other vegetables bao phủ 228 giỏ và có thể là lựa chọn thực tế hơn cho một thử nghiệm gợi ý mua kèm. Đây là dữ liệu quan sát tại một cửa hàng. Cần thử nghiệm để đo tác động kinh doanh.

## Visual Pattern Mining

Chạy ứng dụng ảnh từ đầu trong [notebook Visual Pattern Mining](visual_patterns.ipynb). Notebook này tải ảnh, tạo visual words và dùng Apriori-TID của nhóm.